In [ ]:
!pip install pymupdf
!pip install pypdf
!pip install faiss-cpu # Added to install faiss
!pip install llama-index

In [ ]:
import json
import os
import io
import re
#import requests
import dotenv
import transformers
import pypdf
import faiss
#import sqlite3

import fitz  # PyMuPDF

from dotenv import load_dotenv

from openai import OpenAI
from sentence_transformers import util, SentenceTransformer
from transformers import pipeline, BertTokenizer, BertModel

import pandas as pd
import numpy as np
from io import StringIO
from dotenv import load_dotenv
#from operator import itemgetter

import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')

from llama_index.core.node_parser import SentenceSplitter

# **Setting up Python Environment**

## Instructions for setting up your .env file:

1. Create a .env file in the same directory as this notebook

2. Add the following lines to the .env file:

    OPENAI_API_KEY=<your_openai_api_key>

    HF_TOKEN=<your_huggingface_token>

3. Replace the placeholders with your actual keys.

4. Save the file.

5. Restart the kernel to ensure the keys are loaded correctly.

# Load API Keys into the Notebook Environment:

In [ ]:
load_dotenv()

openai = os.getenv('OPENAI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

OPENAI_API_KEY = openai
HF_TOKEN = HF_TOKEN
print(HF_TOKEN)
print(OPENAI_API_KEY)

# Custom Functions for Chunking the CMU Student Handbook & Measuring Computational Cost
(Optional / not required for the homework that you use these functions)

In [ ]:
# Splitting Text into Sentences
def split_text_into_sentences_v1(text):
    sentences = nltk.sent_tokenize(text)
    return sentences

def split_text_into_sentences_v2(text):
    sentences = sent_tokenize(text, language='english')  # Default is usually 'english'
    return sentences

In [ ]:
# define a function to split the resumes into sentences and assign unique identifiers:

def split_resumes_to_sentences(df, text_column):
    """
    Split the resumes into individual sentences and assign unique identifiers.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the resumes.
        text_column (str): The name of the column containing the resume texts.

    Returns:
        pd.DataFrame: A DataFrame with each sentence and its corresponding unique identifier.
    """
    # Initialize an empty list to hold the resulting data
    sentences_list = []

    # Iterate through the DataFrame rows
    for idx, row in df.iterrows():
        # Tokenize the resume text into sentences
        sentences = sent_tokenize(row[text_column])

        # Append each sentence along with the original index to the list
        for sentence in sentences:
            sentences_list.append((idx, sentence))

    # Convert the list to a DataFrame
    sentences_df = pd.DataFrame(sentences_list, columns=['unique_identifier', 'sentence'])

    return sentences_df

In [ ]:
# import time
# from sklearn.cluster import DBSCAN

# def compute_embedding_costs(text, model_name='all-MiniLM-L6-v2', eps=0.6, min_samples=2):
#     """
#     Computes the computational cost (in terms of execution time) for creating
#     sentence embeddings and paraphrase embeddings.

#     Parameters:
#     - text (str): The input text to be processed.
#     - model_name (str): The name of the model to use for embedding.
#     - eps (float): The epsilon value for DBSCAN clustering.
#     - min_samples (int): The minimum sample count for DBSCAN clustering.

#     Returns:
#     - A tuple containing the execution times for sentence embeddings and paraphrase-level embeddings.
#     """
#     # Note: A UserWarning regarding HF_TOKEN not being in Colab secrets may appear.
#     # This is because huggingface_hub expects tokens to be stored in Colab secrets
#     # for authentication, even if the token is set as an environment variable or is not
#     # strictly required for public models like 'all-MiniLM-L6-v2'.
#     model = SentenceTransformer(model_name)

#     # Sentence Embedding Timing
#     start_time = time.time()
#     sentences = sent_tokenize(text)
#     sentence_embeddings = model.encode(sentences)
#     sentence_embedding_time = time.time() - start_time

#     # Paraphrase Embedding (Clustering) Timing
#     start_clustering_time = time.time()
#     clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine').fit(sentence_embeddings)
#     cluster_labels = clustering.labels_

#     paraphrase_embeddings = []
#     for cluster_id in set(cluster_labels):
#         if cluster_id == -1:
#             continue
#         cluster_sentences = np.array(sentences)[cluster_labels == cluster_id]
#         paraphrase = ' '.join(cluster_sentences)
#         paraphrase_embeddings.append(paraphrase)
#     paraphrase_embedding_time = time.time() - start_clustering_time

#     return sentence_embedding_time, paraphrase_embedding_time

# # Example usage
# if __name__ == "__main__":
#     text = ("This is a sample text. It has several sentences, meant to showcase "
#             "how embeddings are computed. Some of these sentences may be clustered "
#             "together, representing paraphrases or semantically similar groups.")

#     sent_time, para_time = compute_embedding_costs(text)
#     print(f"Sentence Embedding Time: {sent_time:.4f} seconds")
#     print(f"Paraphrase Embedding Time: {para_time:.4f} seconds")

# # This function was created with GenerativeAI Assistance.

In [ ]:
# import torch

# def estimate_model_flops(model_name, text):
#     """
#     Estimate the FLOPs for generating embeddings for a given text using a specified model.

#     Parameters:
#     - model_name (str): Model identifier from Hugging Face Transformers.
#     - text (str): Text to process.

#     Returns:
#     - FLOPs (int): An estimated number of floating point operations.
#     """
#     tokenizer = BertTokenizer.from_pretrained(model_name)
#     model = BertModel.from_pretrained(model_name)

#     inputs = tokenizer(text, return_tensors="pt")
#     input_ids = inputs['input_ids']

#     # Hooks for the operations
#     def hook_fn_forward(module, input, output):
#         # Attempt to access the tensor shape in a safer manner
#         input_shape = input[0].size()

#         # A generalized fallback if shape isn't what's expected
#         if len(input_shape) == 2:  # Assuming shape [batch, seq_len] for simplicity
#             batch_size, seq_len = input_shape
#             # Hypothetical FLOPs calculation: For demonstration, let's assume it's just the product
#             flops = batch_size * seq_len
#         elif len(input_shape) > 2:  # Assuming more dimensions (e.g., embeddings)
#             flops = torch.prod(torch.tensor(input_shape))
#         else:
#             # In case of unsupported dimensions, set flops to 0 or some placeholder
#             flops = 0

#         # Storing calculated FLOPs in the module
#         if hasattr(module, '__flops__'):
#             module.__flops__ += flops
#         else:
#             module.__flops__ = flops

#     def add_hooks_to_model(model, hook_fn):
#         """
#         Recursively add hook_fn to all the layers of the model.
#         """
#         total_flops = 0
#         for layer in model.children():
#             if list(layer.children()):  # if the layer has children, recursively add hooks
#                 total_flops += add_hooks_to_model(layer, hook_fn)
#             else:
#                 if hasattr(layer, 'weight'):
#                     layer.register_forward_hook(hook_fn)
#                     layer.__flops__ = 0
#         return total_flops

#     add_hooks_to_model(model, hook_fn_forward)

#     with torch.no_grad():
#         _ = model(**inputs)

#     total_flops = sum([mod.__flops__ for mod in model.modules() if hasattr(mod, '__flops__')])

#     return total_flops

# # Example usage
# if __name__ == "__main__":
#     model_name = "bert-base-uncased"
#     text = "This is an example sentence"
#     flops = estimate_model_flops(model_name, text)
#     print(f"Estimated FLOPs: {flops}")

# # This function was created with GenerativeAI Assistance.

# Load Data into the Notebook Environment:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
doc = fitz.open("path/to/cmu-student-policy-handbook.pdf")
doc

# **Homework 2 Assignment**

## **Section A. Experimenting with Vector Store Query Design (50 points)**

### **Choose a method to chunk the text data:**

- [Semantic chunking](https://python.langchain.com/docs/modules/data_connection/document_transformers/semantic-chunker)

- [Recursive chunking](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter)

- [Character chunking](https://python.langchain.com/docs/modules/data_connection/document_transformers/character_text_splitter)

- [Token chunking](https://python.langchain.com/docs/modules/data_connection/document_transformers/split_by_token)

# Sentence Chunking

In [ ]:
# PARAGRAPH-BASED CHUNKING WITH OVERLAP
# This creates better results than sentence-based chunking for long documents

from llama_index.core.node_parser import SentenceSplitter

# Create a paragraph splitter with overlap
# chunk_size = 512 characters (about 2-3 paragraphs)
# chunk_overlap = 50 characters (some overlap to maintain context)
text_parser = SentenceSplitter(
    chunk_size=512,      # Size of each chunk in characters
    chunk_overlap=50     # Number of overlapping characters between chunks
)

# Initialize lists to store chunks and their IDs
text_chunks = []
doc_idxs = []

# Process each page of the PDF
print("📚 Chunking the document...")
for doc_idx, page in enumerate(doc):
    # Extract text from the page
    page_text = page.get_text("text")

    # Split the page text into chunks
    cur_text_chunks = text_parser.split_text(page_text)

    # Add chunks to our list
    text_chunks.extend(cur_text_chunks)

    # Keep track of which page each chunk came from
    doc_idxs.extend([doc_idx] * len(cur_text_chunks))

    if (doc_idx + 1) % 50 == 0:  # Progress update every 50 pages
        print(f"  Processed {doc_idx + 1} pages...")

print(f"\n Chunking complete!")
print(f" Total chunks created: {len(text_chunks)}")
print(f" Average chunk size: {np.mean([len(chunk) for chunk in text_chunks]):.0f} characters")

In [ ]:
# Create a DataFrame to organize our chunks
# This makes it easy to work with the data
text_chunk_df = pd.DataFrame({
    'page_number': doc_idxs,
    'text': text_chunks
})

print(f" DataFrame created with {len(text_chunk_df)} chunks")
print(f"\n Preview of first chunk:")
print(text_chunk_df.iloc[0]['text'][:200] + "...")

# **Choose an embedding model to use for creating embeddings of the text chunks and create the Embeddings**

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

In [ ]:
# Create embeddings for all chunks
print(" Creating embeddings for all chunks...")

chunk_embeddings = model.encode(
    text_chunk_df['text'].tolist(),
    show_progress_bar=True
)

print(f" Embedding dimensions: {chunk_embeddings.shape}")
print(f"   - Number of chunks: {chunk_embeddings.shape[0]}")
print(f"   - Dimensions per embedding: {chunk_embeddings.shape[1]}")

# **Create a FAISS Vector Store**

In [ ]:
# Create a FAISS Vector Store using DOT PRODUCT similarity
# (IndexFlatIP = Index Flat Inner Product)

# Get the dimension of our embeddings
dimension = chunk_embeddings.shape[1]

print(f" Building vector store with Dot Product similarity...")
print(f"   Dimension: {dimension}")

# IMPORTANT: For Dot Product (Inner Product), we need to normalize embeddings
# This makes them unit length so dot product = cosine similarity
from sklearn.preprocessing import normalize
chunk_embeddings_normalized = normalize(chunk_embeddings, norm='l2')

# Create FAISS index with Inner Product (Dot Product) similarity
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product

# Add our normalized embeddings to the index
index.add(chunk_embeddings_normalized.astype('float32'))

print(f" Vector store created!")
print(f" Total vectors in store: {index.ntotal}")

# Homework requirement:

# Section A

## **Query the vector store using these queries**

**Instruction: set the 'k' parameter to 5**

Query 1: What is the policy statement for the academic integrity policy?

Query 2: What is the policy violation definition for cheating?

Query 3: What is the policy statement for improper or illegal communications?

Query 4: What are CMU’s quiet hours?

Query 5: Where are pets allowed on CMU?


### ***query the vector store with the 5 queries above (don't forget to record the responses in your homework submission spreadsheet: see instructions for a link to the spreadsheet!):***

In [ ]:
# Function to query the vector store and get results
def query_vector_store(question, k=5):
    """
    Query the vector store and return k most similar chunks

    Parameters:
    - question: Your query as a string
    - k: Number of results to retrieve (default=5)

    Returns:
    - results_df: DataFrame with results
    """
    # Create embedding for the question
    query_embedding = model.encode([question])

    # Normalize the query embedding (same as we did for chunk embeddings)
    query_embedding_normalized = normalize(query_embedding, norm='l2')

    # Search the vector store
    # D = distances (similarity scores), I = indices of matching chunks
    D, I = index.search(query_embedding_normalized.astype('float32'), k)

    # Get the matching chunks
    results = []
    for idx, (similarity, chunk_idx) in enumerate(zip(D[0], I[0])):
        results.append({
            'rank': idx + 1,
            'similarity_score': float(similarity),
            'chunk_text': text_chunk_df.iloc[chunk_idx]['text'],
            'page_number': text_chunk_df.iloc[chunk_idx]['page_number'] + 1  # +1 for human-readable
        })

    results_df = pd.DataFrame(results)
    return results_df


# THE 5 REQUIRED QUERIES FOR PART A
queries = [
    "What is the policy statement for the academic integrity policy?",
    "What is the policy violation definition for cheating?",
    "What is the policy statement for improper or illegal communications?",
    "What are CMU's quiet hours?",
    "Where are pets allowed on CMU?"
]

# Store all results
all_results = {}

print("🔍 Running all 5 required queries...\n")

for i, query in enumerate(queries, 1):
    print(f"Query {i}: {query}")
    print("=" * 80)

    # Query with k=5 (as required by assignment)
    results = query_vector_store(query, k=5)

    # Store results
    all_results[f"Query_{i}"] = {
        'query_text': query,
        'results': results
    }

    # Display results
    print(f"\nTop 5 Results:\n")
    for _, row in results.iterrows():
        print(f"Rank {row['rank']} (Similarity: {row['similarity_score']:.4f}, Page: {row['page_number']})")
        print(f"Text: {row['chunk_text'][:150]}...")
        print()

    print("\n" + "="*80 + "\n")

print("✅ All queries completed!")

In [ ]:
# Prepare data for the spreadsheet template
spreadsheet_data = []

for query_num in range(1, 6):
    query_key = f"Query_{query_num}"
    query_info = all_results[query_key]

    # For each result chunk (1-5)
    for response_num, row in query_info['results'].iterrows():
        spreadsheet_data.append({
            'Section': 'Part A',
            'Query #': query_num,
            'Query Text': query_info['query_text'],
            'k parameter': 5,
            'Response #': response_num + 1,
            'Response Text': row['chunk_text']
        })

# Create DataFrame
submission_df = pd.DataFrame(spreadsheet_data)

# Save to CSV
submission_df.to_csv('part_a_results.csv', index=False)

print("✅ Results saved to 'part_a_results.csv'")
print(f"📊 Total rows: {len(submission_df)}")
print("\n📋 You can now copy this data to the Google Spreadsheet template!")

# **Homework Questions:**

**A.I.**

(i) Describe these distance metrics: Cosine similarity; Euclidean Distance; Dot Product.

(ii) For each of the metrics you defined in (i), describe how the metric is different from the other metrics.

(iii) For each of the metrics you defined in (i), describe one advantage and one disadvantage of using the metric.



**A.II.** Copy and paste the results or information retrieved from the vector store in response to each of the queries you submitted to the vector store in the SPREADSHEET TEMPLATE (please see instructions for a link to the spreadsheet template you should copy and use).  


**A.III.** Qualitatively analyze the responses to your queries submitted to the vector store. Did the queries retrieve the information you were expecting to obtain. Why or why not? Why do you think the queries were successful / unsuccessful in retrieving the information you expected or needed?

# **Section B. Experimenting with Vector Store Embeddings & Query Parameters (50 points)**

1) Choose 1 of the 5 queries provided in Section A, above, and experiment with submitting the query to the vector store by changing the QUERY and RETRIEVAL_NUMBER parameters in the following manner:


*   A) Baseline query (e.g. query), retrieval_number parameter=1.

*   B) Query, retrieval_number parameter  = 3

*   C) Query, retrieval_number parameter  = 5

*   D) Query, retrieval_number parameter  = 10

**In your written homework submission, record the UNIQUE responses/results of each query submitted to the vector store.**

2. Select a different text chunking method (e.g. word, sentence, paragraph) and:
   
- Chunk your text data using the method.
- Create embeddings for the text.
- Load the embeddings into the vector store.
- Submit the same query you selected in B.1, above, and submit it to the vector store 6 times (using the different ‘retrieval_number’ parameter settings defined in B.1, above), and record the responses.

**In your written homework submission, record the responses/results of each query submitted to the vector store.**

### **Homework Questions:**

**B.I.** Explain your rationale for selecting the query you choose in B.1. Why did you choose this query vs. the other 4 queries?

**B.II.** Copy and paste the responses to the queries you submitted to the vector store in the SPREADSHEET TEMPLATE.

**B.III.** Copy and paste the responses to the queries you submitted to the vector store in the SPREADSHEET TEMPLATE.

**B.IV.** In observing the responses from the vector store to the queries created in B.1., which ‘k’ parameter do you think retrieved the highest quality / most accurate result? Why do you think this parameter was the best to use with the query?

**B.V.** In observing the responses from the vector store to the queries created in B.2., which ‘k’ parameter do you think retrieved the highest quality / most accurate result? Why do you think this parameter was the best to use with the query?

# **BONUS TASKS / QUESTIONS: Define function to call LLM API**

## Please email Sara for the Bonus Task Python Notebook once you've completed your homework assignment